In [ ]:
# Standard Block
if i % 2 == 0:
    shock = 0.02
else:
    shock = -0.01

# Ternary Shorthand
shock = 0.02 if (i % 2 == 0) else -0.01

In [ ]:
# Standard Block
squared = []
for x in range(5):
    squared.append(x**2)

# Shorthand
squared = [x**2 for x in range(5)]

In [ ]:
returns = [-0.05, 0.12, -0.01, 0.04]

# Read: Zero out negative returns, keep positive returns
floored_returns = [0.0 if r < 0 else r for r in returns]
# Output: [0.0, 0.12, 0.0, 0.04]

##### Dictionary Comprehensions

Purpose: Map assets, tickers, or keys to values efficiently.

In [ ]:
tickers = ["AAPL", "MSFT", "TSLA"]

# Shorthand to initialize a blank weights dictionary
portfolio = {ticker: 0.0 for ticker in tickers}
# Output: {'AAPL': 0.0, 'MSFT': 0.0, 'TSLA': 0.0}

In [ ]:
# Raw values from database
raw_exposure = {"AAPL": 100_000, "MSFT": 250_000, "TSLA": 80_000}

# Shorthand to apply mathematical transformation on the fly
# .items() gives us both the Key (k) and the Value (v) at the same time
haircut_exposure = {k: v * 0.90 for k, v in raw_exposure.items()}

print(haircut_exposure)
# Output: {'AAPL': 90000.0, 'MSFT': 225000.0, 'TSLA': 72000.0}

##### Lambda Functions (Anonymous Functions)

Purpose: Throwaway, single-line mathematical transformations or key sorting logic.

In [ ]:
# Standard Block
def linear_scale(x):
    return x * 100

# Shorthand
linear_scale = lambda x: x * 100

##### Generator/`yield`

Purpose: Creates a Generator. Instead of creating a massive array/list that hogs RAM, it calculates and feeds data one item at a time (lazy evaluation).

In [ ]:
# 1. LIST COMPREHENSION (Creates a massive list in RAM immediately)
prices_list = [p**2 for p in range(10_000_000)]

# 2. GENERATOR EXPRESSION (The Shorthand! Uses zero RAM)
prices_stream = (p**2 for p in range(10_000_000))

# Shorthand with transformation and condition
total_risk = sum(p * 0.05 for p in historical_returns if p > 0)

# Maps negatives to zero, keeps positives as is.
stream = (x if x >= 0 else 0.0 for x in named_list)

# Keeps only the items > 100. Drops everything else.
stream = (x for x in named_list if x > 100)

In [ ]:
# Our named list of historical returns
portfolio_returns = [0.02, -0.05, 0.01, -0.03, 0.04, -0.01]

# Shorthand with a trailing 'if' condition
# Read: Multiply by 100, FOR every r in returns, IF r is less than 0
loss_stream = (r * 100 for r in portfolio_returns if r < 0)

# Execute the stream
for loss_pct in loss_stream:
    print(f"Analyzing loss day: {loss_pct:.1f}%")

Analyzing loss day: -5.0%
Analyzing loss day: -3.0%
Analyzing loss day: -1.0%


##### Scenario 1: State Tracking (e.g., Random Walks / Monte Carlo)

In risk management, you rarely stream independent, isolated numbers. You usually stream path-dependent data—where today's simulated price depends entirely on yesterday's simulated price.

You cannot do this with a shorthand because a shorthand cannot remember its own previous output. You must use yield:

In [ ]:
import random

def stream_random_walk(start_price: float, steps: int):
    """Simulates a stock price path over time."""
    current_price = start_price

    for _ in range(steps):
        # 1. Calculate a complex random shock
        shock = random.normalvariate(0, 0.02)

        # 2. Update the internal state (Path Dependency)
        current_price = current_price * (1 + shock)

        # 3. Stream the current state out
        yield current_price

##### Scenario 2: Error Handling & Defensive Code

If you are streaming real-world tick data from an external network socket or a database, things fail. Production-grade data pipelines require try/except blocks to handle network timeouts or malformed data strings.

You cannot wrap a shorthand in a try/except block line-by-line.

In [ ]:
def stream_live_market_data(api_connection):
    while True:
        try:
            raw_packet = api_connection.receive_data()
            parsed_price = float(raw_packet['price'])
            yield parsed_price
        except KeyError:
            print("Malformed packet received, skipping...")
            continue
        except ConnectionResetError:
            print("Reconnecting to exchange...")
            break

##### Scenario 3: Resource Management (with blocks)
When streaming data out of massive multi-gigabyte files, you need to open the file safely and ensure it closes automatically when the stream finishes. 

Shorthand cannot safely manage file open/close states on its own.

In [ ]:
def stream_csv_rows(file_path: str):
    # The generator safely controls the file lifetime
    with open(file_path, 'r') as file:
        for line in file:
            yield line.strip().split(',')
    # File automatically closes here when the generator finishes or errors out